# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaihanBasha7/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
# Install required packages
!pip -q install datasets duckdb pyarrow pandas huggingface_hub

In [25]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:1000]",
    token=HF_TOKEN
)

print(dataset)
print(dataset.features)
print(dataset[0])

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 1000
})
{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessio

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

**One row means:** One content item's daily search and analytics performance for a specific client on a specific report date.

**Table used:** `fact_content_daily_performance`

**Time window:** The analysis uses the first available portion of the `fact_content_daily_performance` table loaded from the warehouse for schema verification and contract validation. In a full analysis, a mid-panel month such as March 2026 would be preferred to avoid using the final evaluation period.

**Prediction target:** Predict future content performance using historical Search Console and GA4 metrics.

**Excluded:** Future information and label-derived fields are excluded because they are unavailable at prediction time and would introduce data leakage.

In [26]:
import pandas as pd

df = dataset.to_pandas()

print("Total rows loaded:", len(df))
print("Date range:")
print(df["report_date"].min(), "to", df["report_date"].max())

grain_duplicates = (
    df.groupby(["report_date", "client_hash_id", "content_hash_id"])
      .size()
      .reset_index(name="count")
)

duplicates = grain_duplicates[grain_duplicates["count"] > 1]

print("Duplicate grain rows:", len(duplicates))
duplicates.head()

Total rows loaded: 1000
Date range:
2025-01-27 to 2025-01-30
Duplicate grain rows: 0


,report_date,client_hash_id,content_hash_id,count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

These metrics are available before making a prediction.

### Label / Proxy
Future search performance (or a future performance label created later).

### Context
- report_date
- client_hash_id
- content_hash_id

These identify observations and are used for grouping, joining, and analysis but not as model inputs.

### Excluded
- Future-derived labels
- Future metrics
- Hash IDs as features

These are excluded to prevent data leakage or meaningless model learning.

In [27]:
print("Columns available:\n")
for c in df.columns:
    print("-", c)

Columns available:

- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

The following checks verify the data contract.

1. Confirm the row grain.
2. Count rows and inspect the date window.
3. Check how many rows contain valid GA4 data.

In [28]:
print("Total rows:", len(df))

print("\nDate window:")
print(df["report_date"].min())
print(df["report_date"].max())

print("\nGSC available rows:")
print(df["gsc_data_available"].value_counts())

print("\nGA4 available rows:")
print(df["ga4_data_available"].value_counts())

print("\nRows with GA4 available:")
print(df[df["ga4_data_available"] == True].shape[0])

print("\nMissing values:")
print(df.isnull().sum())

Total rows: 1000

Date window:
2025-01-27
2025-01-30

GSC available rows:
gsc_data_available
True    1000
Name: count, dtype: int64

GA4 available rows:
ga4_data_available
False    1000
Name: count, dtype: int64

Rows with GA4 available:
0

Missing values:
report_date                 0
client_hash_id              0
content_hash_id             0
client_has_gsc              0
client_has_ga4              0
gsc_data_available          0
ga4_data_available          0
gsc_impressions             0
gsc_clicks                  0
gsc_sum_position            0
gsc_avg_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
sessions_organic            0
sessions_direct             0
sessions_referral           0
sessions_social             0
sessions_paid               0
sessions_ai                 0
ai_chatgpt                  0
ai_perplexity               0
ai_gemini              

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset supports search-performance analysis but has important limitations.

- Client history lengths vary.
- GA4 metrics before availability are zero-filled.
- Search Console and GA4 coverage differ between clients.
- Hash IDs cannot be interpreted as meaningful features.
- This analysis identifies observed patterns and supports decisions, but it cannot establish causal relationships.

In [29]:
print("Clients with GSC:")
print(df["client_has_gsc"].value_counts())

print("\nClients with GA4:")
print(df["client_has_ga4"].value_counts())

print("\nGA4 data availability:")
print(df["ga4_data_available"].value_counts())

Clients with GSC:
client_has_gsc
True    1000
Name: count, dtype: int64

Clients with GA4:
client_has_ga4
True    1000
Name: count, dtype: int64

GA4 data availability:
ga4_data_available
False    1000
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Five Feature Frame

| Feature | Knowable at prediction time because... |
|----------|-----------------------------------------|
| gsc_impressions | Historical Search Console metric available before prediction |
| gsc_clicks | Historical search performance |
| gsc_avg_position | Historical ranking signal |
| ga4_pageviews | Historical user engagement |
| ga4_sessions | Historical traffic metric when GA4 data is available |

## Leakage Demonstration

If a future-derived label or future performance metric is accidentally included as a feature, the model can appear unrealistically accurate. Such features leak future information and must be removed before training. Only information available at the prediction moment should be used.